# Jordan Lake Spatiotemporal Simulation
ASV wind characterization simulation at the Jordan Lake mission domain.

Features:
- STGPKF Formulation
    - Generate Synthetic Data
    - Make estimates based upon existing measurements
- Ergodic Control
- Real-Time SOC-Based speed control
- Known Hyperparameters

In [ ]:
# Import Packages
using Logging: global_logger
using TerminalLoggers: TerminalLogger
global_logger(TerminalLogger())

using ProgressLogging

using LinearAlgebra
using Random
using Statistics
using Plots
using LaTeXStrings
using Revise
using StaticArrays
using Interpolations
using LazySets
using SpatiotemporalGPs
using JLD2
using LinearInterpolations

In [ ]:
# Import modules
include("../src/jordan_lake_domain.jl")
# include("../src/stgpkf.jl")
include("../src/kf.jl")
include("../src/ngpkf.jl")
include("../src/SyntheticData.jl")
include("../src/ergodic.jl")
include("../src/variograms.jl")
include("../src/SOC_Controller.jl")
include("../src/simulator_spatial.jl")
include("../src/simulator_ST.jl")
include("../src/Convex_bound_avoidance.jl")

# Synthetic Data

In [1]:
# Define simulation time-scale/steps
Δt = 2.5 # seconds
dt_min = Δt/(60) # convert ΔT from seconds to minutes
dt_hrs = Δt/(60*60) # convert ΔT from seconds to hours
T_begin = 9.0; # hours
T_end = 12.0; # hours 
ts_hrs = T_begin:dt_hrs:T_end # hours
ts_min = T_begin*60:dt_min:T_end*60

540.0:0.041666666666666664:720.0

In [ ]:
# setup the spatial and temporal kernels
σt = 2.0   # m/s
σs = 1.0   # m/s
lt = 0.25*60.0  # minutes
ls = 0.75   # km

kt = Matern(1/2, σt, lt)
ks = Matern(1/2, σs, ls)

# determine the spatial step size
Δx = 0.10 # km

# create the spatial domain
xs = 0:Δx:1.4
ys = 0:Δx:6.5

grid_pts = vec([@SVector[x, y] for x in xs, y in ys]);

In [ ]:
# generate synthetic data
synthetic_data = STGPKF.generate_spatiotemporal_process(xs, ys, dt_min, (T_end-T_begin)*60, ks, kt);

In [ ]:
synthetic_data.ts

In [ ]:
windfield = @animate for time_idx in 1:100:length(synthetic_data.ts)
    heatmap(synthetic_data.xs, synthetic_data.ys, synthetic_data.data[:, :, time_idx]', clims=(-5,5), cmap=:balance, plottype=:wx)
    title!("Wind at Time = $(Int(floor(ts_hrs[time_idx]))):$(Int(floor(mod(ts_hrs[time_idx],1)*60)))")
    xlabel!("x [km]")
    ylabel!("y [km]")
    # plot!(aspect_ratio=:equal)
end
gif(windfield)

In [ ]:
bounded_windfield = @animate for time_idx in 1:100:length(synthetic_data.ts)
    heatmap(synthetic_data.xs, synthetic_data.ys, synthetic_data.data[:, :, time_idx]', clims=(-5,5), cmap=:balance, plottype=:wx)
    polygon_vertices = hcat(JordanLakeDomain.convex_polygon.vertices, JordanLakeDomain.convex_polygon.vertices[:, 1])  # Close the polygon
    plot!(polygon_vertices[1, :], polygon_vertices[2, :], seriestype=:shape, fillalpha=0.2, label="")
    title!("Wind at Time = $(Int(floor(ts_hrs[time_idx]))):$(Int(floor(mod(ts_hrs[time_idx],1)*60)))")
    xlabel!("x [km]")
    ylabel!("y [km]")
    # plot!(aspect_ratio=:equal)
end
gif(bounded_windfield)

# Known Hyperparameter Simulation

## Controller Setup

In [ ]:
# Create STGPKF Problem
problem = STGPKFProblem(grid_pts, ks, kt, dt_min)
kern = ks
ngp_grid_x = synthetic_data.xs
ngp_grid_y = synthetic_data.ys
ngpkf_grid = NGPKF.NGPKFGrid(ngp_grid_x, ngp_grid_y, kern)

# ASV Start Location
x0s = [@SVector[0.75, 3.0] for i=1:1]

In [ ]:
target_q = 0.95

# Set the rated value matrix    
Nx, Ny = length(synthetic_data.xs), length(synthetic_data.ys)
target_q_mat = ones(Nx, Ny)
target_q_mat *= 0.0

# x_domain = range(0, 1.4, length=Nx)
# y_domain = range(0, 6.5, length=Ny)

x_domain = synthetic_data.xs
y_domain = synthetic_data.ys

for i in 1:length(x_domain)
    for j in 1:length(y_domain)
        p = [x_domain[i], y_domain[j]]
        if p ∈ JordanLakeDomain.convex_polygon.polygon
            target_q_mat[i, j] = 0.95
        end
    end
end

In [ ]:
function Cfun(p, x)
    return kern(x, p)^2 / kern(p, p)
end
function Rfun(p, x)
    return (kern(x,x) - kern(x, p)^2 / kern(p, p) + 0.5^2)/(Δt)
end


S(p, x)  = Cfun(p, x)^2 / Rfun(p, x)
DxS(p, x) = ForwardDiff.gradient(xx-> S(p, xx), x)

In [ ]:
using DifferentialEquations
using ForwardDiff

function clarity_prediction(t, q0, C, R, Q)

    k = C / sqrt(Q * R)
    
    q∞ = k / (1 + k)

    γ1 = q∞ - q0
    γ2 = γ1 * (k-1)
    γ3 = (k-1) * q0 - k

    return q∞ * ( 1 + 2 * γ1 / (γ2 + γ3 * exp(2 * k * Q * t)))
end

    
function clarity_time(q0, qf, C, R, Q; tmax=10.0)
    
    println("q0: $(q0)")
    println("qf: $(qf)")
    println("C: $(C)")
    println("R: $(R)")
    println("Q: $(Q)")
    
    if q0 >= qf
        return 0.0
    end

    k = C / sqrt(Q * R)
    println("k: $(k)")
    
    q∞ = k / (1 + k)
    println("q∞: $(q∞)")
    γ1 = q∞ - q0
    γ2 = γ1 * (k-1)
    γ3 = (k-1) * q0 - k

    
        
    if qf >= q∞
        return tmax
    end

    t = log((2*q∞*γ1 - qf*γ2 + q∞*γ2)/((qf - q∞)*γ3))/(2*k*Q)
    println("t: $(t)")

    return min(t, tmax)
end


C_ = Cfun(0,0)
R_ = Rfun(0,0)

k = (C_^2 / R_)
# q(t) = ((-k * t * q_0) + (k * t * q_0)) / ((-k * t * q_0) + (k * t) + 1)

function Clarity_delta_t(current_clarity, target_clarity)
    delta_t = (target_clarity - current_clarity) / ((target_clarity - 1) * k * (current_clarity - 1))
    return delta_t
end

function Clarity_delta_new(current_clarity, target_clarity)
    den = -target_clarity*k + k*current_clarity*target_clarity + k - k*current_clarity
    return delta_t = (target_clarity - current_clarity) / den
end


In [ ]:
# Controller for weighted 2
using LinearAlgebra, StatsBase

function ergo_controller_weighted_2(t, xs, Mean, w_rated_val, convex_polygon;
        ergo_grid,
        ergo_q_map,
        traj,
        umax= 0.15, #30.0 * 60 / 1000,
        ΔT,
        kwargs...
        )
        
    target_q = 0.95

    # Set the rated value matrix    
    Nx, Ny = length(synthetic_data.xs), length(synthetic_data.ys)
    w_rated = ones(Nx, Ny)
    w_rated *= w_rated_val
  
#   Compute the target matrix 
    lambda_param = 0.05
    delta = -lambda_param*((Mean - w_rated).^2)
    q_target_temp = target_q*(exp.(delta)) 
    
    
    # Mask the matrix such that q_target is zero outisde the domain
    # x_domain = range(0, 1.4, length=Nx)
    # y_domain = range(0, 6.5, length=Ny)
    x_domain = synthetic_data.xs
    y_domain = synthetic_data.ys

    for i in 1:length(x_domain)
        for j in 1:length(y_domain)
            p = [x_domain[i], y_domain[j]]
            if (p ∈ convex_polygon.polygon) == false
                q_target_temp[i, j] = 0.0
            end
        end
    end

    # Get it in the right shape
    q_target_itp = linear_interpolation((x_domain, y_domain), q_target_temp, extrapolation_bc=Interpolations.Line())
    q_target_weighted = q_target_itp(ErgodicController.xs(ergo_grid), ErgodicController.ys(ergo_grid))     

    
    target_spatial_dist = zeros(size(ergo_q_map))
    Qp  = mean(σ_t.^2) # diagm(vec(σ_t .^2 * fuse_measurements_every_ΔT ))      
    
    for i in CartesianIndices(target_spatial_dist)
        if q_target_weighted[i] > ergo_q_map[i]
#             println("here in first cond")
            target_spatial_dist[i] = Clarity_delta_new(ergo_q_map[i], q_target_weighted[i])
        else
            target_spatial_dist[i] = 0.0
        end
    end
        
    u = [ErgodicController.controller_single_integrator_cvx_bound(ergo_grid, x, traj, target_spatial_dist, convex_polygon; umax=umax, do_boundary_correction=true) for x in xs]
    
#     println(u)
    
    return u, q_target_temp

end

ergo_controllers_weighted_2 = [ergo_controller_weighted_2 for i=1:length(x0s)]

## Generate SOC Target Profile

In [ ]:
# Generate SOC_Target Profile
soc_begin = 3000
soc_end = 3500
lcbf = SoCController.compute_lcbf(ts_hrs, dt_hrs);
ucbf = SoCController.compute_ucbf(ts_hrs, dt_hrs);
soc_target = SoCController.generate_SOC_target(lcbf, ucbf,  soc_begin, soc_end, ts_hrs, dt_hrs);
plot(ts_hrs, soc_target, label="SOC Target")
plot!(ts_hrs, lcbf, label="LCBF")
plot!(ts_hrs,ucbf, label="UCBF")

## Simulate

In [ ]:
fuse_measurements_every_ΔT = 5.0/(60) # hours
recompute_controller_every_ΔT = 5.0 / (120.0*60) # minutes
# σ_t = zeros(15, 66);
σ_t = zeros(66, 15);

w_rated_val = 2.75

@time res_ergo_hp =  SimulatorST.simulate_known_param(ts_min, x0s, soc_begin, ergo_controller_weighted_2, soc_target, w_rated_val, JordanLakeDomain.convex_polygon, problem; 
    ngpkf_grid=ngpkf_grid, 
    EnvData=synthetic_data, 
    σ_meas = 0.5,
    # σ_process= 0.075 * fuse_measurements_every_ΔT,
    Q_process = diagm(vec(σ_t .^2 * fuse_measurements_every_ΔT )) , 
    fuse_measurements_every_ΔT = fuse_measurements_every_ΔT, 
    recompute_controller_every_ΔT = recompute_controller_every_ΔT)

In [ ]:
jldsave("20240909_stgpkf_known_hp_sim.jld2"; res_ergo_hp)

# Post Processing

In [ ]:
fuse_measurements_every_ΔT = 5.0/(60*60) # hours
recompute_controller_every_ΔT = 5.0 / (120.0*60) # minutes
# σ_t = zeros(15, 66);
σ_t = zeros(66, 15);

w_rated_val = 2.75
f = jldopen("20240909_stgpkf_known_hp_sim.jld2", "r")

In [ ]:
res_ergo_hp = f["res_ergo_hp"]

In [ ]:
plot()
polygon_vertices = hcat(JordanLakeDomain.convex_polygon.vertices, JordanLakeDomain.convex_polygon.vertices[:, 1])
heatmap(synthetic_data.xs, synthetic_data.ys, synthetic_data.data[:,:,1], cmap = :balance; plottype=:wx)
plot!(polygon_vertices[1, :], polygon_vertices[2, :], seriestype=:shape, fillalpha=0.0, label="", lw = 3, linecolor = "green")
for i=1:length(x0s)
    x = [r[i][1] for r in res_ergo_hp.xs]
    
    y = [r[i][2] for r in res_ergo_hp.xs]
    plot!(x,y, linewidth=3, color=:black, label = "")
end
plot!()

title!("q_T= 0.95 for exponential weighted coverage")

In [ ]:
path_over_real_wind = @animate for n = 1:100:length(res_ergo_hp.ts)
    plot()
    heatmap(synthetic_data.xs, synthetic_data.ys, synthetic_data.data[:,:,n]', cmap = :balance; plottype=:wx)
    plot!(polygon_vertices[1, :], polygon_vertices[2, :], seriestype=:shape, fillalpha=0.0, label="", lw = 2, linecolor = "green")
    for i=1:length(x0s)
        x = [r[i][1] for r in res_ergo_hp.xs[1:n]]
        y = [r[i][2] for r in res_ergo_hp.xs[1:n]]
        plot!(x, y, linewidth=3, color=:black, label="w_rated_val = $(w_rated_val)")
    end
    title!("Path vs Actual Wind Field - Time: $(Int(floor(ts_hrs[n]))):$(Int(floor(mod(ts_hrs[n],1)*60)))")
    plot!(clims=(-5.0,5.0))
end
gif(path_over_real_wind, "../results/full_sim/path_over_real_wind.gif")

In [ ]:
path_over_wind_est = @animate for n = 1:10:length(res_ergo_hp.ts)-1
    plot()
    if res_ergo_hp.ts[n] in res_ergo_hp.w_hat_ts
        global w_hat_idx = findfirst(isequal(res_ergo_hp.ts[n]), res_ergo_hp.w_hat_ts)
    end
    heatmap2 = heatmap(synthetic_data.xs, synthetic_data.ys, res_ergo_hp.w_hats[w_hat_idx]', cmap=:balance)
    plot!(polygon_vertices[1, :], polygon_vertices[2, :], seriestype=:shape, fillalpha=0.0, label="", lw = 2, linecolor = "green")
    for i=1:length(x0s)
        x = [r[i][1] for r in res_ergo_hp.xs[1:n]]
        y = [r[i][2] for r in res_ergo_hp.xs[1:n]]
        plot!(x, y, linewidth=3, color=:black, label="w_rated_val = $(w_rated_val)")
    end
    plot!(clims=(-5.0,5.0))
    # title!("Estimated Wind Field")
    heatmap2 = plot!()
    
    p = plot(heatmap2, plot_title="Path vs Estimated Windfield - Time: $(Int(floor(ts_hrs[n]))):$(Int(floor(mod(ts_hrs[n],1)*60)))")
end
gif(path_over_wind_est, "../results/full_sim/path_over_wind_est.gif")

In [ ]:
update_idxs = [t in res_ergo_hp.w_hat_ts ? 1 : 0 for t in res_ergo_hp.ts]
w_hat_idx = 1

estim_vs_real = @animate for n = 1:10:length(res_ergo_hp.ts)-1
    plot()
    heatmap1 = heatmap(synthetic_data.xs, synthetic_data.ys, synthetic_data.data[:,:,n]', cmap = :balance; plottype=:wx)
    plot!(polygon_vertices[1, :], polygon_vertices[2, :], seriestype=:shape, fillalpha=0.0, label="", lw = 2, linecolor = "green")
    # for i=1:length(x0s)
    #     x = [r[i][1] for r in res_ergo_hp.xs[1:n]]
    #     y = [r[i][2] for r in res_ergo_hp.xs[1:n]]
    #     plot!(x, y, linewidth=3, color=:black, label="w_rated_val = $(w_rated_val)")
    # end
    plot!(clims=(-5.0,5.0))
    title!("Actual Wind Field")
    heatmap1 = plot!()
    if res_ergo_hp.ts[n] in res_ergo_hp.w_hat_ts
        global w_hat_idx = findfirst(isequal(res_ergo_hp.ts[n]), res_ergo_hp.w_hat_ts)
    end
    heatmap2 = heatmap(synthetic_data.xs, synthetic_data.ys, res_ergo_hp.w_hats[w_hat_idx]', cmap=:balance)
    plot!(polygon_vertices[1, :], polygon_vertices[2, :], seriestype=:shape, fillalpha=0.0, label="", lw = 2, linecolor = "green")
    plot!(clims=(-5.0,5.0))
    title!("Estimated Wind Field")
    heatmap2 = plot!()
    
    p = plot(heatmap1, heatmap2, layout=(1,2), plot_title="Time: $(Int(floor(ts_hrs[n]))):$(Int(floor(mod(ts_hrs[n],1)*60)))")
    # plot!(p, title="Path vs Actual Wind Field - Time: $(Int(floor(ts_hrs[n]))):$(Int(floor(mod(ts_hrs[n],1)*60)))")
    # title!("Path vs Actual Wind Field - Time: $(Int(floor(ts_hrs[n]))):$(Int(floor(mod(ts_hrs[n],1)*60)))")
end
gif(estim_vs_real, "../results/full_sim/estim_vs_real.gif")

In [ ]:
update_idxs = [t in res_ergo_hp.w_hat_ts ? 1 : 0 for t in res_ergo_hp.ts]
w_hat_idx = 1
measurement_val = []
estim_val = []

estim_vs_real_path = @animate for n = 1:10:length(res_ergo_hp.ts)-1
    plot()
    heatmap1 = heatmap(synthetic_data.xs, synthetic_data.ys, synthetic_data.data[:,:,n]', cmap = :balance; plottype=:wx)
    plot!(polygon_vertices[1, :], polygon_vertices[2, :], seriestype=:shape, fillalpha=0.0, label="", lw = 2, linecolor = "green")
    for i=1:length(x0s)
        x = [r[i][1] for r in res_ergo_hp.xs[1:n]]
        y = [r[i][2] for r in res_ergo_hp.xs[1:n]]
        plot!(x, y, linewidth=3, color=:black, label="w_rated_val = $(w_rated_val)")
    end
    plot!(clims=(-5.0,5.0))
    title!("Actual Wind Field")
    heatmap1 = plot!()
    if res_ergo_hp.ts[n] in res_ergo_hp.w_hat_ts
        global w_hat_idx = findfirst(isequal(res_ergo_hp.ts[n]), res_ergo_hp.w_hat_ts)
    end
    heatmap2 = heatmap(synthetic_data.xs, synthetic_data.ys, res_ergo_hp.w_hats[w_hat_idx]', cmap=:balance)
    plot!(polygon_vertices[1, :], polygon_vertices[2, :], seriestype=:shape, fillalpha=0.0, label="", lw = 2, linecolor = "green")
    plot!(clims=(-5.0,5.0))
    title!("Estimated Wind Field")
    heatmap2 = plot!()
    
    p = plot(heatmap1, heatmap2, layout=(1,2), plot_title="Time: $(Int(floor(ts_hrs[n]))):$(Int(floor(mod(ts_hrs[n],1)*60)))")


    # Collect measurements
    push!(measurement_val, res_ergo_hp.measurements[n])
    x = res_ergo_hp.xs[n][end][1];
    y = res_ergo_hp.xs[n][end][2];

    itp = linear_interpolation((xs, ys), res_ergo_hp.w_hats[w_hat_idx])
    
    push!(estim_val, itp(x, y))
    # plot!(p, title="Path vs Actual Wind Field - Time: $(Int(floor(ts_hrs[n]))):$(Int(floor(mod(ts_hrs[n],1)*60)))")
    # title!("Path vs Actual Wind Field - Time: $(Int(floor(ts_hrs[n]))):$(Int(floor(mod(ts_hrs[n],1)*60)))")
end
gif(estim_vs_real_path)

In [ ]:
findnearest(A,x) = argmin(abs.(A .- x))

In [ ]:
update_idxs = [t in res_ergo_hp.w_hat_ts ? 1 : 0 for t in res_ergo_hp.ts]
w_hat_idx = 1
measurement_val = []
estim_val = []
estim_dt = 10 # minutes

estim_vs_real_path = @animate for n = 1:10:length(res_ergo_hp.ts)-1
    plot()
    heatmap1 = heatmap(synthetic_data.xs, synthetic_data.ys, synthetic_data.data[:,:,n]', cmap = :balance; plottype=:wx)
    plot!(polygon_vertices[1, :], polygon_vertices[2, :], seriestype=:shape, fillalpha=0.0, label="", lw = 2, linecolor = "green")
    for i=1:length(x0s)
        idx = max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt)
        new_idx = findnearest(res_ergo_hp.ts, idx)
        x = [r[i][1] for r in res_ergo_hp.xs[new_idx:n]]
        y = [r[i][2] for r in res_ergo_hp.xs[new_idx:n]]
        plot!(x, y, linewidth=3, color=:black, label=false)
    end
    # for idx = res_ergo_hp.ts[n]:-0.01:max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt)
    #     # for idx = max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt):0.01:res_ergo_hp.ts[n]
    #     new_idx = findnearest(res_ergo_hp.ts, idx)
    #     x = [r[new_idx][1] for r in res_ergo_hp.xs[1:n]]
    #     y = [r[new_idx][2] for r in res_ergo_hp.xs[1:n]]
    #     plot!(x, y, linewidth=3, color=:black, label="w_rated_val = $(w_rated_val)")
    # end
    plot!(clims=(-5.0,5.0))
    title!("Actual Wind Field")
    heatmap1 = plot!()
    if res_ergo_hp.ts[n] in res_ergo_hp.w_hat_ts
        global w_hat_idx = findfirst(isequal(res_ergo_hp.ts[n]), res_ergo_hp.w_hat_ts)
    end
    heatmap2 = heatmap(synthetic_data.xs, synthetic_data.ys, res_ergo_hp.w_hats[w_hat_idx]', cmap=:balance)
    plot!(polygon_vertices[1, :], polygon_vertices[2, :], seriestype=:shape, fillalpha=0.0, label="", lw = 2, linecolor = "green")
    plot!(clims=(-5.0,5.0))
    title!("Estimated Wind Field")
    heatmap2 = plot!()

    # Clarity Plot
    claritymap = heatmap(synthetic_data.xs, synthetic_data.ys, res_ergo_hp.ergo_q_maps[w_hat_idx]')
    plot!(polygon_vertices[1, :], polygon_vertices[2, :], seriestype=:shape, fillalpha=0.0, label="", lw = 2, linecolor = "green")
    plot!(clims=(0.0,1.0))
    title!("Clarity")
    claritymap = plot!()

    # Error Plot
    push!(measurement_val, res_ergo_hp.measurements[n])
    x = res_ergo_hp.xs[n][end][1];
    y = res_ergo_hp.xs[n][end][2];

    # estim_idx = max(res_ergo_hp.ts[n] - estim_dt, res_ergo_hp.ts[1])
    estim_idx = res_ergo_hp.ts[n]
    new_idx = findnearest(res_ergo_hp.w_hat_ts, estim_idx)
    itp = linear_interpolation((xs, ys), res_ergo_hp.w_hats[new_idx])
    push!(estim_val, itp(x, y))

    errors = []
    for idx = res_ergo_hp.ts[n]:-0.01:max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt)
    # for idx = max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt):0.01:res_ergo_hp.ts[n]
        new_idx = findnearest(res_ergo_hp.w_hat_ts, idx)
        itp = linear_interpolation((xs, ys), res_ergo_hp.w_hats[new_idx])
        push!(errors, abs(res_ergo_hp.measurements[n] - itp(x,y)))
    end
    
    xrange = 0:0.01:res_ergo_hp.ts[n]-max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt)
    errorplot = plot(xrange, errors, label=false)
    ylims!(0, 10)
    title!("Estimation Error")
    xlabel!(L"$\Delta T$ [min]")
    ylabel!("Absolute Error")
    errorplot = plot!()

    p = plot(heatmap1, heatmap2, claritymap, errorplot, layout=(2,2), plot_title="Time: $(Int(floor(ts_hrs[n]))):$(Int(floor(mod(ts_hrs[n],1)*60))) - w_rated_val:$(w_rated_val)")
    # p = plot(heatmap1, heatmap2, claritymap, layout=(1,3), plot_title="Time: $(Int(floor(ts_hrs[n]))):$(Int(floor(mod(ts_hrs[n],1)*60)))")


end
gif(estim_vs_real_path, "../results/full_sim/errorgif.gif")

In [ ]:
plot(measurement_val, label="measurements")
plot!(estim_val, label="estimates")

In [ ]:
point_errors = (measurement_val .- estim_val).^2
t_idxs = 1:10:length(res_ergo_hp.ts)-1
tvals = res_ergo_hp.ts[t_idxs]./60

plot(tvals, point_errors)

In [ ]:
findnearest(A,x) = argmin(abs.(A .- x))

In [ ]:
update_idxs = [t in res_ergo_hp.w_hat_ts ? 1 : 0 for t in res_ergo_hp.ts]
w_hat_idx = 1
measurement_val = []
estim_val = []
estim_dt = 10 # minutes

realtime_errors = @animate for n = 1:10:length(res_ergo_hp.ts)-1
    # Collect measurements
    push!(measurement_val, res_ergo_hp.measurements[n])
    x = res_ergo_hp.xs[n][end][1];
    y = res_ergo_hp.xs[n][end][2];

    # estim_idx = max(res_ergo_hp.ts[n] - estim_dt, res_ergo_hp.ts[1])
    estim_idx = res_ergo_hp.ts[n]
    new_idx = findnearest(res_ergo_hp.w_hat_ts, estim_idx)
    itp = linear_interpolation((xs, ys), res_ergo_hp.w_hats[new_idx])
    push!(estim_val, itp(x, y))

    errors = []
    for idx = res_ergo_hp.ts[n]:-0.01:max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt)
    # for idx = max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt):0.01:res_ergo_hp.ts[n]
        new_idx = findnearest(res_ergo_hp.w_hat_ts, idx)
        itp = linear_interpolation((xs, ys), res_ergo_hp.w_hats[new_idx])
        push!(errors, abs(res_ergo_hp.measurements[n] - itp(x,y)))
    end
    
    xrange = 0:0.01:res_ergo_hp.ts[n]-max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt)
    plot(xrange,errors, label=false)
    ylims!(0, 10)
    title!("Estimation Error at Time: $(Int(floor(ts_hrs[n]))):$(Int(floor(mod(ts_hrs[n],1)*60)))")
    xlabel!(L"$\Delta T$ [min]")
    ylabel!("Absolute Error")
end
gif(realtime_errors, "../results/full_sim/realtime_error.gif")

In [ ]:
kernel(x) = (σt^2) * (1 - exp(-x))
int_kernel(x) = (1 - exp(-x))
subkern(dt, dx) = (dt/lt) + (dx/ls)

In [ ]:
update_idxs = [t in res_ergo_hp.w_hat_ts ? 1 : 0 for t in res_ergo_hp.ts]
w_hat_idx = 1
measurement_val = []
estim_val = []
estim_dt = 10 # minutes

kern_error = @animate for n = 1:10:length(res_ergo_hp.ts)-1
    # Collect measurements
    push!(measurement_val, res_ergo_hp.measurements[n])
    x = res_ergo_hp.xs[n][end][1];
    y = res_ergo_hp.xs[n][end][2];

    # estim_idx = max(res_ergo_hp.ts[n] - estim_dt, res_ergo_hp.ts[1])
    estim_idx = res_ergo_hp.ts[n]
    new_idx = findnearest(res_ergo_hp.w_hat_ts, estim_idx)
    itp = linear_interpolation((xs, ys), res_ergo_hp.w_hats[new_idx])
    push!(estim_val, itp(x, y))

    errors = []
    kern_errors = []
    kern_vals = []
    for idx = res_ergo_hp.ts[n]:-0.01:max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt)
    # for idx = max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt):0.01:res_ergo_hp.ts[n]
        new_idx = findnearest(res_ergo_hp.w_hat_ts, idx)
        itp = linear_interpolation((xs, ys), res_ergo_hp.w_hats[new_idx])
        push!(errors, (res_ergo_hp.measurements[n] - itp(x,y))^2)


        # Get position at which estimate was made
        t_idx = findnearest(res_ergo_hp.ts, res_ergo_hp.w_hat_ts[new_idx])
        x_past = res_ergo_hp.xs[t_idx][end][1]
        y_past = res_ergo_hp.xs[t_idx][end][2]

        dx = sqrt((x - x_past)^2 + (y - y_past)^2);
        dt = abs(res_ergo_hp.ts[n] - res_ergo_hp.w_hat_ts[new_idx]);
        push!(kern_vals, subkern(dx, dt))
        push!(kern_errors, kernel(subkern(dx, dt)))

    end
    
    xrange = 0:0.01:res_ergo_hp.ts[n]-max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt)
    # plot(xrange,errors, label=false)
    scatter(kern_vals, errors)
    plot!(kern_vals, kern_errors)
    ylims!(0, 5)
    title!("Estimation Error at Time: $(Int(floor(ts_hrs[n]))):$(Int(floor(mod(ts_hrs[n],1)*60)))")
    xlabel!(L"$\left( \frac{\Delta t}{l_t} + \frac{\Delta x}{l_x} \right)$")
    ylabel!("Absolute Error")
end
gif(kern_error, "../results/full_sim/kern_error.gif")

In [ ]:
update_idxs = [t in res_ergo_hp.w_hat_ts ? 1 : 0 for t in res_ergo_hp.ts]
w_hat_idx = 1
measurement_val = []
estim_val = []
estim_dt = 10 # minutes


kern_vals = 0:0.01:1
kern_bins = [zeros(1) for _ in 1:length(kern_vals)]
kern_error = @animate for n = 1:10:length(res_ergo_hp.ts)-1
    # Collect measurements
    push!(measurement_val, res_ergo_hp.measurements[n])
    x = res_ergo_hp.xs[n][end][1];
    y = res_ergo_hp.xs[n][end][2];

    # estim_idx = max(res_ergo_hp.ts[n] - estim_dt, res_ergo_hp.ts[1])
    estim_idx = res_ergo_hp.ts[n]
    new_idx = findnearest(res_ergo_hp.w_hat_ts, estim_idx)
    itp = linear_interpolation((xs, ys), res_ergo_hp.w_hats[new_idx])
    push!(estim_val, itp(x, y))

    errors = []
    
    for idx = res_ergo_hp.ts[n]:-0.01:max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt)
    # for idx = max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt):0.01:res_ergo_hp.ts[n]
        new_idx = findnearest(res_ergo_hp.w_hat_ts, idx)
        itp = linear_interpolation((xs, ys), res_ergo_hp.w_hats[new_idx])
        push!(errors, (res_ergo_hp.measurements[n] - itp(x,y))^2)


        # Get position at which estimate was made
        t_idx = findnearest(res_ergo_hp.ts, res_ergo_hp.w_hat_ts[new_idx])
        x_past = res_ergo_hp.xs[t_idx][end][1]
        y_past = res_ergo_hp.xs[t_idx][end][2]

        dx = sqrt((x - x_past)^2 + (y - y_past)^2);
        dt = abs(res_ergo_hp.ts[n] - res_ergo_hp.w_hat_ts[new_idx]);
        kern_val = int_kernel(subkern(dx, dt))
        kern_idx = findnearest(kern_vals, kern_val);
        push!(kern_bins[kern_idx], abs(res_ergo_hp.measurements[n] - itp(x,y)))

    end

  
    xrange = 0:0.01:res_ergo_hp.ts[n]-max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt)
    # plot(xrange,errors, label=false)
    scatter(kern_vals, mean.(kern_bins), label=false)
    # plot!(kern_vals, kernel.(kern_vals), label="Kernel")
    ylims!(0, 5)
    title!("Estimation Error at Time: $(Int(floor(ts_hrs[n]))):$(Int(floor(mod(ts_hrs[n],1)*60)))")
    xlabel!(L"$\left( 1 - e^{- \frac{\Delta t}{l_t} - \frac{\Delta x}{l_x}} \right)$")
    ylabel!("Squared Error")
end
gif(kern_error, "../results/full_sim/kern_error.gif")

In [ ]:
update_idxs = [t in res_ergo_hp.w_hat_ts ? 1 : 0 for t in res_ergo_hp.ts]
w_hat_idx = 1
measurement_val = []
estim_val = []
estim_dt = 10 # minutes


kern_error_reset = @animate for n = 1:10:length(res_ergo_hp.ts)-1
    # Collect measurements
    push!(measurement_val, res_ergo_hp.measurements[n])
    x = res_ergo_hp.xs[n][end][1];
    y = res_ergo_hp.xs[n][end][2];

    # estim_idx = max(res_ergo_hp.ts[n] - estim_dt, res_ergo_hp.ts[1])
    estim_idx = res_ergo_hp.ts[n]
    new_idx = findnearest(res_ergo_hp.w_hat_ts, estim_idx)
    itp = linear_interpolation((xs, ys), res_ergo_hp.w_hats[new_idx])
    push!(estim_val, itp(x, y))

    errors = []
    kern_vals = 0:0.1:15
    kern_bins = [zeros(1) for _ in 1:length(kern_vals)]
    
    for idx = res_ergo_hp.ts[n]:-0.01:max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt)
    # for idx = max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt):0.01:res_ergo_hp.ts[n]
        new_idx = findnearest(res_ergo_hp.w_hat_ts, idx)
        itp = linear_interpolation((xs, ys), res_ergo_hp.w_hats[new_idx])
        push!(errors, (res_ergo_hp.measurements[n] - itp(x,y))^2)


        # Get position at which estimate was made
        t_idx = findnearest(res_ergo_hp.ts, res_ergo_hp.w_hat_ts[new_idx])
        x_past = res_ergo_hp.xs[t_idx][end][1]
        y_past = res_ergo_hp.xs[t_idx][end][2]

        dx = sqrt((x - x_past)^2 + (y - y_past)^2);
        dt = abs(res_ergo_hp.ts[n] - res_ergo_hp.w_hat_ts[new_idx]);
        kern_val = kernel(subkern(dx, dt))
        kern_idx = findnearest(kern_vals, kern_val);
        push!(kern_bins[kern_idx], abs(res_ergo_hp.measurements[n] - itp(x,y)))

    end

  
    xrange = 0:0.01:res_ergo_hp.ts[n]-max(res_ergo_hp.ts[1], res_ergo_hp.ts[n] - estim_dt)
    # plot(xrange,errors, label=false)
    scatter(kern_vals, mean.(kern_bins), label="Data")
    plot!(kern_vals, kernel.(kern_vals), label="Kernel")
    ylims!(0, 5)
    title!("Estimation Error at Time: $(Int(floor(ts_hrs[n]))):$(Int(floor(mod(ts_hrs[n],1)*60)))")
    xlabel!(L"$\left( \frac{\Delta t}{l_t} + \frac{\Delta x}{l_x} \right)$")
    ylabel!("Absolute Error")
end
gif(kern_error_reset, "../results/full_sim/kern_error_reset.gif")

In [ ]:
plot(measurement_val, label="measurements")
plot!(estim_val, label="estimates")

In [ ]:
point_errors = (measurement_val .- estim_val).^2
t_idxs = 1:10:length(res_ergo_hp.ts)-1
tvals = res_ergo_hp.ts[t_idxs]./60

plot(tvals, point_errors)

In [ ]:
# Collect speeds
u1 = Float64[]
u2 = Float64[]
speed = Float64[]

for i=1:length(res_ergo_hp.us)
    push!(u1, res_ergo_hp.us[i][1][1])
    push!(u2, res_ergo_hp.us[i][1][2])
    push!(speed, norm(res_ergo_hp.us[i][1]))
end

# Speed vs time plot
plot(ts_hrs, res_ergo_hp.speeds, label="Speed Controller Command")
plot!(ts_hrs, speed, label="Executed Speed")
ylims!(0., 2.5)
title!("Speed vs Time")
xlabel!("Time [hr, 12 = noon]")
ylabel!("Speed [m/s]")
plot!(legend=:right)

In [ ]:
plot(ts_hrs, res_ergo_hp.bs[2:end], label="SOC")
plot!(ts_hrs, soc_target, label="SOC Target")

In [ ]:
plot(ts_hrs, res_ergo_hp.bs[2:end], label="SOC")
plot!(ts_hrs, soc_target, label="SOC Target")
ylims!(0., 6500)